In [1]:
import config
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\moham\anaconda3\envs\Langchain_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
chat_template_tools = ChatPromptTemplate.from_template(''' 
What are the five most important tools a {job title} needs?
Answer only by listing the tools here.
''')
chat_template_tools #Notice that by default, this new method assumes that the template we passed as an argument is a human message prompt template. This will ease our implementation even more.

ChatPromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, template=' \nWhat are the five most important tools a {job title} needs?\nAnswer only by listing the tools here.\n'), additional_kwargs={})])

In [3]:
chat_template_strategy = ChatPromptTemplate.from_template(''' 
Considering the tools provided, develop a strategy for effectively learning and mastering them: {tools}
''')

In [4]:
chat = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature = 0,
    seed = 365,
    max_tokens = 30,
    openai_api_key = config.api_key
)

In [5]:
string_parser =  StrOutputParser()

In [6]:
chain_tools = chat_template_tools | chat | string_parser | {'tools': RunnablePassthrough()}
chain_strategy = chat_template_strategy | chat | string_parser

In [9]:
chain_long = (chat_template_tools | chat | string_parser | {'tools': RunnablePassthrough()} |
                     chat_template_strategy | chat | string_parser
)

In [10]:
chain_long.get_graph()

Graph(nodes={'79d94c9058c845adbb4aceb1748e9de6': Node(id='79d94c9058c845adbb4aceb1748e9de6', name='PromptInput', data=<class 'langchain_core.utils.pydantic.PromptInput'>, metadata=None), 'd07103030bf24822b8e73455081316f1': Node(id='d07103030bf24822b8e73455081316f1', name='ChatPromptTemplate', data=ChatPromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, template=' \nWhat are the five most important tools a {job title} needs?\nAnswer only by listing the tools here.\n'), additional_kwargs={})]), metadata=None), '98f5c5af76974dba83062a83ace71a46': Node(id='98f5c5af76974dba83062a83ace71a46', name='ChatOpenAI', data=ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': Fal

In [12]:
chain_long.get_graph().print_ascii() #library that allowed for visualizing chain nodes and their sequence through an Ascii print.

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       